In [1]:
# ==========================
# Imports
# ==========================
import os
import tensorflow as tf
from sklearn.model_selection import train_test_split
import pandas as pd

# ==========================
# Config
# ==========================
IMG_HEIGHT = 224
IMG_WIDTH = 224
BATCH_SIZE = 32
EPOCHS = 30            # scratch → train longer
DATA_DIR = './dataset'  # cataract, diabetic_retinopathy, glaucoma, normal

# ==========================
# Read dataset
# ==========================
data = []
classes = sorted(os.listdir(DATA_DIR))
for cls in classes:
    cls_path = os.path.join(DATA_DIR, cls)
    for img_name in os.listdir(cls_path):
        img_path = os.path.join(cls_path, img_name)
        if os.path.isfile(img_path):
            data.append((img_path, cls))

df = pd.DataFrame(data, columns=['image_path', 'label'])
print(f"Total images: {len(df)}")
print(f"Classes found: {df['label'].unique()}")

# Encode
label_to_index = {label: idx for idx, label in enumerate(classes)}
df['label_idx'] = df['label'].map(label_to_index)
NUM_CLASSES = len(classes)

# Split
train_df, val_df = train_test_split(
    df, 
    test_size=0.2, 
    stratify=df['label_idx'], 
    random_state=42
)

# ==========================
# Data augmentation
# ==========================
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.15),
    tf.keras.layers.RandomZoom(0.15),
])

# ==========================
# Dataset preprocessing
# ==========================
def preprocess_image(file_path, label):
    img = tf.io.read_file(file_path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, [IMG_HEIGHT, IMG_WIDTH])
    img = img / 255.0
    label = tf.one_hot(label, NUM_CLASSES)
    return img, label

def df_to_dataset(dataframe, shuffle=True, batch_size=BATCH_SIZE):
    paths = dataframe['image_path'].values
    labels = dataframe['label_idx'].values
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(buffer_size=len(dataframe))
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

train_dataset = df_to_dataset(train_df)
val_dataset   = df_to_dataset(val_df, shuffle=False)

# ==========================
# CNN FROM SCRATCH
# ==========================
def make_cnn(input_shape=(IMG_HEIGHT, IMG_WIDTH, 3), num_classes=NUM_CLASSES):

    inputs = tf.keras.Input(shape=input_shape)

    x = data_augmentation(inputs)

    # Block 1
    x = tf.keras.layers.Conv2D(32, (3,3), activation='relu', padding='same')(x)
    x = tf.keras.layers.Conv2D(32, (3,3), activation='relu', padding='same')(x)
    x = tf.keras.layers.MaxPooling2D(2)(x)

    # Block 2
    x = tf.keras.layers.Conv2D(64, (3,3), activation='relu', padding='same')(x)
    x = tf.keras.layers.Conv2D(64, (3,3), activation='relu', padding='same')(x)
    x = tf.keras.layers.MaxPooling2D(2)(x)

    # Block 3
    x = tf.keras.layers.Conv2D(128, (3,3), activation='relu', padding='same')(x)
    x = tf.keras.layers.Conv2D(128, (3,3), activation='relu', padding='same')(x)
    x = tf.keras.layers.MaxPooling2D(2)(x)

    # Block 4
    x = tf.keras.layers.Conv2D(256, (3,3), activation='relu', padding='same')(x)
    x = tf.keras.layers.Conv2D(256, (3,3), activation='relu', padding='same')(x)
    x = tf.keras.layers.MaxPooling2D(2)(x)

    x = tf.keras.layers.Flatten()(x)
    x = tf.keras.layers.Dropout(0.5)(x)
    x = tf.keras.layers.Dense(256, activation='relu')(x)
    x = tf.keras.layers.Dropout(0.5)(x)

    outputs = tf.keras.layers.Dense(num_classes, activation='softmax')(x)

    return tf.keras.Model(inputs, outputs)

model = make_cnn()
model.summary()

# ==========================
# Compile
# ==========================
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='AUC')]
)

# ==========================
# Train
# ==========================
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=EPOCHS
)



2025-12-06 09:36:33.448622: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1765010193.461947  630126 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1765010193.466018  630126 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1765010193.476005  630126 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1765010193.476015  630126 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1765010193.476017  630126 computation_placer.cc:177] computation placer alr

Total images: 4217
Classes found: ['cataract' 'diabetic_retinopathy' 'glaucoma' 'normal']


I0000 00:00:1765010195.558916  630126 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 8244 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3080, pci bus id: 0000:06:00.0, compute capability: 8.6


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential (Sequential)         │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 224, 224, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 224, 224, 32)   │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 112, 112, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 112, 112, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 56, 56, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 56, 56, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, 28, 28, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 28, 28, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 14, 14, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 50176)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 50176)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │    12,845,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │         1,028 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

...

 Trainable params: 14,018,596 (53.48 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/30


I0000 00:00:1765010204.901188  630204 cuda_dnn.cc:529] Loaded cuDNN version 90300


106/106 ━━━━━━━━━━━━━━━━━━━━ 18s 62ms/step - AUC: 0.6301 - accuracy: 0.3596 - loss: 1.3219 - val_AUC: 0.7178 - val_accuracy: 0.4467 - val_loss: 1.2227
Epoch 2/30
106/106 ━━━━━━━━━━━━━━━━━━━━ 11s 52ms/step - AUC: 0.7372 - accuracy: 0.4681 - loss: 1.1838 - val_AUC: 0.7916 - val_accuracy: 0.5509 - val_loss: 1.0903
Epoch 3/30
106/106 ━━━━━━━━━━━━━━━━━━━━ 12s 52ms/step - AUC: 0.8029 - accuracy: 0.5458 - loss: 1.0469 - val_AUC: 0.8200 - val_accuracy: 0.5865 - val_loss: 1.0274
Epoch 4/30
106/106 ━━━━━━━━━━━━━━━━━━━━ 11s 51ms/step - AUC: 0.8262 - accuracy: 0.5734 - loss: 0.9905 - val_AUC: 0.8669 - val_accuracy: 0.6410 - val_loss: 0.8819
Epoch 5/30
106/106 ━━━━━━━━━━━━━━━━━━━━ 11s 51ms/step - AUC: 0.8553 - accuracy: 0.6178 - loss: 0.9079 - val_AUC: 0.8818 - val_accuracy: 0.6659 - val_loss: 0.8398
Epoch 6/30
106/106 ━━━━━━━━━━━━━━━━━━━━ 11s 52ms/step - AUC: 0.8754 - accuracy: 0.6404 - loss: 0.8411 - val_AUC: 0.8887 - val_accuracy: 0.6635 - val_loss: 0.7997
Epoch 7/30
106/106 ━━━━━━━━━━━━━━━━━━━━

In [2]:
history_fine = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=10
)


Epoch 1/10
106/106 ━━━━━━━━━━━━━━━━━━━━ 11s 52ms/step - AUC: 0.9760 - accuracy: 0.8621 - loss: 0.3641 - val_AUC: 0.9521 - val_accuracy: 0.7950 - val_loss: 0.5376
Epoch 2/10
106/106 ━━━━━━━━━━━━━━━━━━━━ 11s 52ms/step - AUC: 0.9741 - accuracy: 0.8577 - loss: 0.3781 - val_AUC: 0.9252 - val_accuracy: 0.7263 - val_loss: 0.7075
Epoch 3/10
106/106 ━━━━━━━━━━━━━━━━━━━━ 11s 52ms/step - AUC: 0.9757 - accuracy: 0.8553 - loss: 0.3687 - val_AUC: 0.9625 - val_accuracy: 0.8116 - val_loss: 0.4618
Epoch 4/10
106/106 ━━━━━━━━━━━━━━━━━━━━ 11s 52ms/step - AUC: 0.9760 - accuracy: 0.8565 - loss: 0.3642 - val_AUC: 0.9571 - val_accuracy: 0.8081 - val_loss: 0.5042
Epoch 5/10
106/106 ━━━━━━━━━━━━━━━━━━━━ 11s 52ms/step - AUC: 0.9771 - accuracy: 0.8627 - loss: 0.3548 - val_AUC: 0.9732 - val_accuracy: 0.8519 - val_loss: 0.3895
Epoch 6/10
106/106 ━━━━━━━━━━━━━━━━━━━━ 11s 52ms/step - AUC: 0.9784 - accuracy: 0.8713 - loss: 0.3421 - val_AUC: 0.9555 - val_accuracy: 0.8033 - val_loss: 0.5079
Epoch 7/10
106/106 ━━━━━━━━━

In [3]:
history_fine = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=10,batch_size=32
)


Epoch 1/10
106/106 ━━━━━━━━━━━━━━━━━━━━ 11s 51ms/step - AUC: 0.9803 - accuracy: 0.8731 - loss: 0.3292 - val_AUC: 0.9517 - val_accuracy: 0.7915 - val_loss: 0.5377
Epoch 2/10
106/106 ━━━━━━━━━━━━━━━━━━━━ 11s 52ms/step - AUC: 0.9775 - accuracy: 0.8698 - loss: 0.3539 - val_AUC: 0.9540 - val_accuracy: 0.7998 - val_loss: 0.5107
Epoch 3/10
106/106 ━━━━━━━━━━━━━━━━━━━━ 11s 52ms/step - AUC: 0.9796 - accuracy: 0.8743 - loss: 0.3340 - val_AUC: 0.9647 - val_accuracy: 0.8211 - val_loss: 0.4499
Epoch 4/10
106/106 ━━━━━━━━━━━━━━━━━━━━ 11s 52ms/step - AUC: 0.9808 - accuracy: 0.8793 - loss: 0.3207 - val_AUC: 0.9460 - val_accuracy: 0.7773 - val_loss: 0.5733
Epoch 5/10
106/106 ━━━━━━━━━━━━━━━━━━━━ 11s 52ms/step - AUC: 0.9806 - accuracy: 0.8793 - loss: 0.3225 - val_AUC: 0.9759 - val_accuracy: 0.8626 - val_loss: 0.3703
Epoch 6/10
106/106 ━━━━━━━━━━━━━━━━━━━━ 11s 52ms/step - AUC: 0.9808 - accuracy: 0.8761 - loss: 0.3208 - val_AUC: 0.9635 - val_accuracy: 0.8211 - val_loss: 0.4582
Epoch 7/10
106/106 ━━━━━━━━━

In [4]:
history_fine = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=10,batch_size=64
)


Epoch 1/10
106/106 ━━━━━━━━━━━━━━━━━━━━ 11s 52ms/step - AUC: 0.9823 - accuracy: 0.8865 - loss: 0.3097 - val_AUC: 0.9761 - val_accuracy: 0.8495 - val_loss: 0.3717
Epoch 2/10
106/106 ━━━━━━━━━━━━━━━━━━━━ 11s 52ms/step - AUC: 0.9824 - accuracy: 0.8823 - loss: 0.3078 - val_AUC: 0.9640 - val_accuracy: 0.8282 - val_loss: 0.4616
Epoch 3/10
106/106 ━━━━━━━━━━━━━━━━━━━━ 11s 52ms/step - AUC: 0.9838 - accuracy: 0.8867 - loss: 0.2950 - val_AUC: 0.9784 - val_accuracy: 0.8685 - val_loss: 0.3553
Epoch 4/10
106/106 ━━━━━━━━━━━━━━━━━━━━ 11s 52ms/step - AUC: 0.9836 - accuracy: 0.8844 - loss: 0.2967 - val_AUC: 0.9629 - val_accuracy: 0.8092 - val_loss: 0.4678
Epoch 5/10
106/106 ━━━━━━━━━━━━━━━━━━━━ 11s 52ms/step - AUC: 0.9831 - accuracy: 0.8870 - loss: 0.3023 - val_AUC: 0.9665 - val_accuracy: 0.8306 - val_loss: 0.4432
Epoch 6/10
106/106 ━━━━━━━━━━━━━━━━━━━━ 11s 52ms/step - AUC: 0.9840 - accuracy: 0.8876 - loss: 0.2933 - val_AUC: 0.9616 - val_accuracy: 0.8246 - val_loss: 0.4705
Epoch 7/10
106/106 ━━━━━━━━━